In [ ]:
import dataiku
import pandas as pd

In [ ]:
client = dataiku.api_client()
users = client.list_users()
user_data = []

In [ ]:
for user_info in users:
    user = client.get_user(user_info['login'])

    user_data.append({
        "Display name": user_info.get('displayName', ''),
        "Last successful login": str(user.get_activity().last_successful_login),
        "Last session activity": str(user.get_activity().last_session_activity),
        'Profile': user_info.get('userProfile', ''),
        'Trial Status': user_info.get('trialStatus', ''),
        "Groups": ", ".join(user_info.get('groups', [])),
        "User creation date": str(user.get_settings().creation_date),
        "Type": user_info.get('sourceType', ''),
        "Email": user_info.get('email', 'N/A'),
        "Login name": user_info.get('login', ''),
    })

In [ ]:
df = pd.DataFrame(user_data).sort_values(by=["Last session activity", "Last successful login"], ascending=False)

In [ ]:
df["Last successful login"] = pd.to_datetime(df["Last successful login"], errors='coerce').dt.tz_localize(None)
df["Last session activity"] = pd.to_datetime(df["Last session activity"], errors='coerce').dt.tz_localize(None)
now = pd.Timestamp.now().tz_localize(None)
df["Most recent activity"] = df[["Last successful login", "Last session activity"]].max(axis=1)
df["Time since last activity"] = now - df["Most recent activity"]

In [ ]:
def format_timedelta(td: "pd.Timedelta | None") -> str:
    """Convert a pandas Timedelta to a human-readable string (e.g., '2 days, 3 hours')."""
    if pd.isna(td):
        return "N/A"
    seconds = int(td.total_seconds())
    days, seconds = divmod(seconds, 86400)
    hours, seconds = divmod(seconds, 3600)
    minutes, seconds = divmod(seconds, 60)

    parts = []
    if days > 0:
        parts.append(f"{days} day{'s' if days != 1 else ''}")
    if hours > 0:
        parts.append(f"{hours} hour{'s' if hours != 1 else ''}")
    if minutes > 0:
        parts.append(f"{minutes} minute{'s' if minutes != 1 else ''}")
    if not parts:
        parts.append(f"{seconds} second{'s' if seconds != 1 else ''}")

    return ", ".join(parts)

df["Time since last activity"] = df["Time since last activity"].apply(format_timedelta)

In [ ]:
df = df[~df['Login name'].str.endswith('@dataiku.com', na=False)]

In [ ]:
priority_columns = ["Display name", "Time since last activity"]
other_columns = [col for col in df.columns if col not in priority_columns]
df = df[priority_columns + other_columns]
df